In [2]:
from attention import trainer, trainer_PCA_comp_brute_force, trainer_PCA_comp_2_model
from attetion_sep_size_heads import trainer_multidomain_strategyB

import os
import torch
import random
import numpy as np

cwd = os.getcwd()
def save_tensor_to_txt(tensor, filename):
    with open(filename, 'w') as f:
        # Write tensor dimensions
        dims = tensor.size()
        f.write(" ".join(map(str, dims)) + "\n")

        # Iterate over the first dimension (slices)
        for i in range(dims[0]):
            f.write("\n")
            f.write(f"Slice {i + 1}\n")
            for j in range(dims[1]):  # Iterate over the second dimension (rows)
                row = tensor[i, j].tolist()
                f.write(",".join(map(str, row)) + "\n")
cwd=cwd.replace(r"\CODE\AttentionDCA_python\src",'')

nb_bins_PCA=35
H = 64
d= 10
n_epochs = 500 #could go more down
#domain1_end = 63 #if your protein msa input family has a domain division, this is the zero index of the last aminoacid of the first domain
domain1_end = 62
filename = cwd + r'\CODE\DataAttentionDCA\jdoms\jdoms_bacteria_train2.fasta' #new lisa for energy couplings: HKRR174
structfile = None
trainer_type = 'std' # 'std' or 'std_with_masks' 'multidomain'
family = 'jdoms'

if trainer_type == 'std':
    domain1_end = 0
    H1 = H2 = 0


if trainer_type == 'std' or trainer_type == 'std_with_masks':
    
    model = trainer(
        n_epochs=n_epochs,
        H=H,
        d=d,
        filename=filename,
        structfile=structfile,
        losstype='without_J',
        index_last_domain1=domain1_end,  # this value is the 0-index include the domain 1, for HK-RR is 63 (so 64 long domain 1) 
        #it is set to zero if i dont want to divide any domain
        H1 = H1,
        H2 = H2,
        max_gap_frac=0.8
    )

    # Create results directory
    simul_name = f'{H}_{d}_{family}_without_J_{n_epochs}_mod_alessandro'
    results_dir = f'./results/{simul_name}'
    os.makedirs(results_dir, exist_ok=True)

    # Save model parameters
    save_tensor_to_txt(model.Q.data, "./results/"+simul_name+"/Q_tensor.txt")
    save_tensor_to_txt(model.K.data, "./results/"+simul_name+"/K_tensor.txt")
    save_tensor_to_txt(model.V.data, "./results/"+simul_name+"/V_tensor.txt")

Total sequences read: 14502
Sequences after filtering: 14502
Sampling 100000 pairs out of 105146751 total pairs.
Mean fraction of identical positions (sampled): 0.37143082557715734
Computed theta: 0.32738262854475986


100%|██████████| 14502/14502 [00:09<00:00, 1575.38it/s]


3265.70225762244
63
Using device: cpu
Epoch 1 - Train Loss: 911.2161, Val Loss: 414.2419
Epoch 2 - Train Loss: 598.8293, Val Loss: 294.5401
Epoch 3 - Train Loss: 446.1712, Val Loss: 221.2850
Epoch 4 - Train Loss: 360.6122, Val Loss: 176.3363
Epoch 5 - Train Loss: 308.1983, Val Loss: 147.8123
Epoch 6 - Train Loss: 273.4130, Val Loss: 128.0126
Epoch 7 - Train Loss: 248.7726, Val Loss: 113.4479
Epoch 8 - Train Loss: 230.1482, Val Loss: 102.0666
Epoch 9 - Train Loss: 215.4368, Val Loss: 92.9470
Epoch 10 - Train Loss: 203.5088, Val Loss: 85.4233
Epoch 11 - Train Loss: 193.4394, Val Loss: 79.0493
Epoch 12 - Train Loss: 185.0449, Val Loss: 73.5931
Epoch 13 - Train Loss: 177.7974, Val Loss: 68.8748
Epoch 14 - Train Loss: 171.4651, Val Loss: 64.7440
Epoch 15 - Train Loss: 165.8228, Val Loss: 61.1350
Epoch 16 - Train Loss: 160.9790, Val Loss: 57.9553
Epoch 17 - Train Loss: 156.5963, Val Loss: 55.0624
Epoch 18 - Train Loss: 152.6811, Val Loss: 52.4774
Epoch 19 - Train Loss: 149.1016, Val Loss: 50

In [1]:
from attention import trainer, trainer_PCA_comp_brute_force, trainer_PCA_comp_2_model
from attetion_sep_size_heads import trainer_multidomain_strategyB

import os
import torch
import random
import numpy as np
import sys
import h5py
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..','src'))
sys.path.insert(0, parent_dir)
from model import AttentionModel
from model_PCA_correlation import AttentionModel_PCA
from dcascore import *
from utils import read_fasta_alignment, remove_duplicate_sequences, add_PCA_coords

# back to original path (in PLM)
sys.path.pop(0)  # Removes the parent_dir from sys.path
def read_tensor_from_txt(filename):
    """
        Usual method to get Q, K, V tensors from text files
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
    # Read the dimensions from the first line
    dims = list(map(int, lines[0].strip().split()))
    
    tensor_data = []
    current_slice = []
    for line in lines[1:]:
        line = line.strip()
        if line.startswith("Slice"):
            if current_slice:
                tensor_data.append(current_slice)
                current_slice = []
        elif line:
            current_slice.append(list(map(float, line.split(','))))
    if current_slice:
        tensor_data.append(current_slice)

    tensor = torch.tensor(tensor_data).view(*dims)
    return tensor



cwd = os.getcwd()
def save_tensor_to_txt(tensor, filename):
    with open(filename, 'w') as f:
        # Write tensor dimensions
        dims = tensor.size()
        f.write(" ".join(map(str, dims)) + "\n")

        # Iterate over the first dimension (slices)
        for i in range(dims[0]):
            f.write("\n")
            f.write(f"Slice {i + 1}\n")
            for j in range(dims[1]):  # Iterate over the second dimension (rows)
                row = tensor[i, j].tolist()
                f.write(",".join(map(str, row)) + "\n")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

family = 'jdoms'
H = 64
d= 10
N = 174
n_epochs = 500
nb_PCA_comp=30
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_mod_alessandro/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_mod_alessandro/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_mod_alessandro/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
Q_1=Q_1.to(device)
K_1=K_1.to(device)
V_1=V_1.to(device)
cwd=cwd.replace(r"\CODE\AttentionDCA_python\src",'')

nb_bins_PCA=35
H = 64
d= 10
n_epochs = 250 #could go more down
#domain1_end = 63 #if your protein msa input family has a domain division, this is the zero index of the last aminoacid of the first domain
domain1_end = 62
filename = cwd + r'\CODE\DataAttentionDCA\jdoms\jdoms_bacteria_train2.fasta' #new lisa for energy couplings: HKRR174
structfile = None
trainer_type = 'std' # 'std' or 'std_with_masks' 'multidomain'


if trainer_type == 'std':
    domain1_end = 0
    H1 = H2 = 0


if trainer_type == 'std' or trainer_type == 'std_with_masks':
    
    model = trainer_PCA_comp_2_model(
        n_epochs,
        Q_1,
        K_1,
        V_1,
        H=H,
        d=d,
        filename=filename,
        structfile=structfile,
        losstype='without_J',
        index_last_domain1=domain1_end,  # this value is the 0-index include the domain 1, for HK-RR is 63 (so 64 long domain 1) 
        #it is set to zero if i dont want to divide any domain
        H1 = H1,
        H2 = H2,
        max_gap_frac=0.8,
        n_comp_pca=nb_PCA_comp
    )

    # Create results directory
    simul_name = f'{H}_{d}_{family}_without_J_{n_epochs}_2model_pretrained_n_pca_{nb_PCA_comp}'
    results_dir = f'./results/{simul_name}'
    os.makedirs(results_dir, exist_ok=True)

    # Save model parameters
    save_tensor_to_txt(model.Q.data, "./results/"+simul_name+"/Q_tensor.txt")
    save_tensor_to_txt(model.K.data, "./results/"+simul_name+"/K_tensor.txt")
    save_tensor_to_txt(model.V.data, "./results/"+simul_name+"/V_tensor.txt")

Total sequences read: 14502
Sequences after filtering: 14502
Sampling 100000 pairs out of 105146751 total pairs.
Mean fraction of identical positions (sampled): 0.37174746031746037
Computed theta: 0.3271037814116


100%|██████████| 14502/14502 [00:09<00:00, 1582.95it/s]


3265.70225762244
Using device: cuda
not working


c:\Users\youss\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\autograd\graph.py:823: UserWarning: Error detected in BmmBackward0. No forward pass information available. Enable detect anomaly during forward pass for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_anomaly_mode.cpp:105.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


OutOfMemoryError: CUDA out of memory. Tried to allocate 9.46 GiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 9.50 GiB is allocated by PyTorch, and 11.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from attention import trainer, trainer_PCA_comp_brute_force, trainer_PCA_comp_2_model, trainer_PCA_comp_2_model_once
from attetion_sep_size_heads import trainer_multidomain_strategyB

import os
import torch
import random
import numpy as np

cwd = os.getcwd()
def save_tensor_to_txt(tensor, filename):
    with open(filename, 'w') as f:
        # Write tensor dimensions
        dims = tensor.size()
        f.write(" ".join(map(str, dims)) + "\n")

        # Iterate over the first dimension (slices)
        for i in range(dims[0]):
            f.write("\n")
            f.write(f"Slice {i + 1}\n")
            for j in range(dims[1]):  # Iterate over the second dimension (rows)
                row = tensor[i, j].tolist()
                f.write(",".join(map(str, row)) + "\n")
cwd=cwd.replace(r"\CODE\AttentionDCA_python\src",'')

nb_bins_PCA=35
n_comp_pca=4
H = 64#64
H_PCA=32
d= 10
n_epochs = 500 #could go more down
#domain1_end = 63 #if your protein msa input family has a domain division, this is the zero index of the last aminoacid of the first domain
domain1_end = 62
filename = cwd + r'\CODE\DataAttentionDCA\jdoms\jdoms_bacteria_train2.fasta' #new lisa for energy couplings: HKRR174
structfile = None
trainer_type = 'std' # 'std' or 'std_with_masks' 'multidomain'
family = 'jdoms'

if trainer_type == 'std':
    domain1_end = 0
    H1 = H2 = 0


if trainer_type == 'std' or trainer_type == 'std_with_masks':
    
    model = trainer_PCA_comp_2_model_once(
        n_epochs=n_epochs,
        H=H,
        d1=d,
        H_PCA=H_PCA,
        d2=d,
        filename=filename,
        structfile=structfile,
        losstype='without_J',
        index_last_domain1=domain1_end,  # this value is the 0-index include the domain 1, for HK-RR is 63 (so 64 long domain 1) 
        #it is set to zero if i dont want to divide any domain
        H1 = H1,
        H2 = H2,
        max_gap_frac=0.8,
        eta=0.03,
        nb_bins_PCA=nb_bins_PCA,n_comp_PCA=n_comp_pca
    )

    # Create results directory
    simul_name = f'{H}_{d}_{family}_without_J_{n_epochs}_n_comp_{n_comp_pca}_PCA_2models_once_{nb_bins_PCA}_bins_fixed_masks'
    results_dir = f'./results/{simul_name}'
    os.makedirs(results_dir, exist_ok=True)

    # Save model parameters
    save_tensor_to_txt(model.Q.data, "./results/"+simul_name+"/Q_tensor.txt")
    save_tensor_to_txt(model.K.data, "./results/"+simul_name+"/K_tensor.txt")
    save_tensor_to_txt(model.V.data, "./results/"+simul_name+"/V_tensor.txt")
    save_tensor_to_txt(model.Q_PCA.data, "./results/"+simul_name+"/Q_tensor_PCA.txt")
    save_tensor_to_txt(model.K_PCA.data, "./results/"+simul_name+"/K_tensor_PCA.txt")
    save_tensor_to_txt(model.V_PCA.data, "./results/"+simul_name+"/V_tensor_PCA.txt")

Total sequences read: 14502
Sequences after filtering: 14502
Sampling 100000 pairs out of 105146751 total pairs.
Mean fraction of identical positions (sampled): 0.37189590769949255
Computed theta: 0.3269732134247035


100%|██████████| 14502/14502 [00:09<00:00, 1572.95it/s]


3265.70225762244
Using device: cpu
Epoch 1 - Train Loss: 712.6721, Val Loss: 230.4684
Epoch 2 - Train Loss: 148.6106, Val Loss: 77.0939
Epoch 3 - Train Loss: 51.3453, Val Loss: 31.5314
Epoch 4 - Train Loss: 23.9438, Val Loss: 17.0009
Epoch 5 - Train Loss: 14.0192, Val Loss: 10.9668
Epoch 6 - Train Loss: 9.6171, Val Loss: 7.9175
Epoch 7 - Train Loss: 7.2167, Val Loss: 6.1073
Epoch 8 - Train Loss: 5.6943, Val Loss: 4.8710
Epoch 9 - Train Loss: 4.6212, Val Loss: 3.9646
Epoch 10 - Train Loss: 3.8178, Val Loss: 3.2702
Epoch 11 - Train Loss: 3.1944, Val Loss: 2.7236
Epoch 12 - Train Loss: 2.6994, Val Loss: 2.2851
Epoch 13 - Train Loss: 2.2997, Val Loss: 1.9285
Epoch 14 - Train Loss: 1.9732, Val Loss: 1.6355
Epoch 15 - Train Loss: 1.7039, Val Loss: 1.3929
Epoch 16 - Train Loss: 1.4804, Val Loss: 1.1909
Epoch 17 - Train Loss: 1.2938, Val Loss: 1.0216
Epoch 18 - Train Loss: 1.1370, Val Loss: 0.8791
Epoch 19 - Train Loss: 1.0048, Val Loss: 0.7585
Epoch 20 - Train Loss: 0.8927, Val Loss: 0.6561
E

In [ ]:
from attention import trainer, trainer_PCA_comp_2_model_flat
from attetion_sep_size_heads import trainer_multidomain_strategyB

import os
import torch
import random
import numpy as np

cwd = os.getcwd()
def save_tensor_to_txt(tensor, filename):
    with open(filename, 'w') as f:
        # Write tensor dimensions
        dims = tensor.size()
        f.write(" ".join(map(str, dims)) + "\n")

        # Iterate over the first dimension (slices)
        for i in range(dims[0]):
            f.write("\n")
            f.write(f"Slice {i + 1}\n")
            for j in range(dims[1]):  # Iterate over the second dimension (rows)
                row = tensor[i, j].tolist()
                f.write(",".join(map(str, row)) + "\n")
cwd=cwd.replace("/CODE/AttentionDCA_python/src",'')

nb_bins_PCA=35
H = 64
d= 10

n_epochs = 500 #could go more down
#domain1_end = 63 #if your protein msa input family has a domain division, this is the zero index of the last aminoacid of the first domain
domain1_end = 62
filename = cwd + '/CODE/DataAttentionDCA/jdoms/jdoms_bacteria_train2.fasta' #new lisa for energy couplings: HKRR174
structfile = None
trainer_type = 'std' # 'std' or 'std_with_masks' 'multidomain'
family = 'jdoms'

if trainer_type == 'std':
    domain1_end = 0
    H1 = H2 = 0


if trainer_type == 'std' or trainer_type == 'std_with_masks':
    
    model = trainer_PCA_comp_2_model_flat(
        n_epochs=n_epochs,
        H=H,
        d=d,
        filename=filename,
        structfile=structfile,
        losstype='without_J',
        index_last_domain1=domain1_end,  # this value is the 0-index include the domain 1, for HK-RR is 63 (so 64 long domain 1) 
        #it is set to zero if i dont want to divide any domain
        H1 = H1,
        H2 = H2,
        max_gap_frac=0.8,
        nb_bins_PCA=nb_bins_PCA
    )

    # Create results directory
    simul_name = f'{H}_{d}_{family}_without_J_{n_epochs}_PCA_2models_flat_{nb_bins_PCA}_bins'
    results_dir = f'./results/{simul_name}'
    os.makedirs(results_dir, exist_ok=True)

    # Save model parameters
    save_tensor_to_txt(model.Q.data, "./results/"+simul_name+"/Q_tensor.txt")
    save_tensor_to_txt(model.K.data, "./results/"+simul_name+"/K_tensor.txt")
    save_tensor_to_txt(model.V.data, "./results/"+simul_name+"/V_tensor.txt")

Total sequences read: 14502
Sequences after filtering: 14502
Sampling 100000 pairs out of 105146751 total pairs.
Mean fraction of identical positions (sampled): 0.37189095300163066
Computed theta: 0.3269775696841617


100%|██████████| 14502/14502 [00:09<00:00, 1486.92it/s]


3265.70225762244
Using device: cpu
not working
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
Epoch 1 - Train Loss: 73761.2992, Val Loss: 45958.6198
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
Epoch 2 - Train Loss: 32030.4849, Val Loss: 19482.5560
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
Epoch 3 - Train Loss: 13514.9996, Val Loss: 8199.4368
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
Epoch 4 - Train Loss: 5691.9171, Val Loss: 3460.0531
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
Epoch 5 - Train Loss: 2404.0533, Val Loss: 1468.2622
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
N2:  1
Epoch 6 - Train Loss: 1021.2553, Val Loss: 629.0251
N

In [ ]:
from attention import trainer, trainer_PCA_comp_2_model_flat,trainer_PCA_comp_2_model_test_cross
from attetion_sep_size_heads import trainer_multidomain_strategyB

import os
import torch
import random
import numpy as np

cwd = os.getcwd()
def save_tensor_to_txt(tensor, filename):
    with open(filename, 'w') as f:
        # Write tensor dimensions
        dims = tensor.size()
        f.write(" ".join(map(str, dims)) + "\n")

        # Iterate over the first dimension (slices)
        for i in range(dims[0]):
            f.write("\n")
            f.write(f"Slice {i + 1}\n")
            for j in range(dims[1]):  # Iterate over the second dimension (rows)
                row = tensor[i, j].tolist()
                f.write(",".join(map(str, row)) + "\n")
cwd=cwd.replace("/CODE/AttentionDCA_python/src",'')

nb_bins_PCA=35
H = 64
d= 10
n_epochs = 500 #could go more down
#domain1_end = 63 #if your protein msa input family has a domain division, this is the zero index of the last aminoacid of the first domain
domain1_end = 62
filename = cwd + '/CODE/DataAttentionDCA/jdoms/jdoms_bacteria_train2.fasta' #new lisa for energy couplings: HKRR174
structfile = None
trainer_type = 'std' # 'std' or 'std_with_masks' 'multidomain'
family = 'jdoms'

if trainer_type == 'std':
    domain1_end = 0
    H1 = H2 = 0


if trainer_type == 'std' or trainer_type == 'std_with_masks':
    
    model = trainer_PCA_comp_2_model_test_cross(
        n_epochs=n_epochs,
        H=H,
        d=d,
        filename=filename,
        structfile=structfile,
        losstype='without_J',
        index_last_domain1=domain1_end,  # this value is the 0-index include the domain 1, for HK-RR is 63 (so 64 long domain 1) 
        #it is set to zero if i dont want to divide any domain
        H1 = H1,
        H2 = H2,
        max_gap_frac=0.8,
        nb_bins_PCA=nb_bins_PCA
    )

    # Create results directory
    simul_name = f'{H}_{d}_{family}_without_J_{n_epochs}_PCA_2models_cross_loss_test_{nb_bins_PCA}_bins'
    results_dir = f'./results/{simul_name}'
    os.makedirs(results_dir, exist_ok=True)

    # Save model parameters
    save_tensor_to_txt(model.Q.data, "./results/"+simul_name+"/Q_tensor.txt")
    save_tensor_to_txt(model.K.data, "./results/"+simul_name+"/K_tensor.txt")
    save_tensor_to_txt(model.V.data, "./results/"+simul_name+"/V_tensor.txt")